<a href="https://colab.research.google.com/github/Mbokade2004/Mbokade2004/blob/main/Secure_e_commerce_system_for_handling_payment_using_python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =========================
# Secure E-Commerce (Python, Colab-ready)
# Features: SQLite, PBKDF2 password hashing, card tokenization, HMAC "gateway", Luhn validation
# UI: Gradio
# =========================

# 1) Setup & Imports
!pip -q install gradio==4.44.0

import os, sqlite3, hashlib, hmac, secrets, base64, re, time, json
from datetime import datetime
import gradio as gr

DB_PATH = "ecommerce_secure.db"

# Secrets (rotate for production)
APP_SECRET = secrets.token_bytes(32)             # used for session/HMAC in app
GATEWAY_SECRET = secrets.token_bytes(32)         # used to sign "gateway" payloads

# 2) Helpers: Password hashing, Card validation/tokenization, DB utils
def pbkdf2_hash(password: str, salt: bytes=None, iters: int=200_000):
    if salt is None:
        salt = secrets.token_bytes(16)
    dk = hashlib.pbkdf2_hmac("sha256", password.encode("utf-8"), salt, iters, dklen=32)
    return base64.b64encode(salt).decode(), base64.b64encode(dk).decode(), iters

def pbkdf2_verify(password: str, salt_b64: str, hash_b64: str, iters: int):
    salt = base64.b64decode(salt_b64.encode())
    dk2 = hashlib.pbkdf2_hmac("sha256", password.encode("utf-8"), salt, iters, dklen=32)
    return hmac.compare_digest(base64.b64encode(dk2).decode(), hash_b64)

def luhn_check(card_number: str) -> bool:
    num = re.sub(r"\D", "", card_number)
    if not num or len(num) < 12:  # demo len guard
        return False
    total = 0
    rev = num[::-1]
    for i, ch in enumerate(rev):
        d = ord(ch) - 48
        if i % 2 == 1:
            d *= 2
            if d > 9:
                d -= 9
        total += d
    return total % 10 == 0

def validate_expiry(mm_yy: str) -> bool:
    # Accept MM/YY or MM/YYYY
    m = re.match(r"^\s*(\d{2})\s*/\s*(\d{2}|\d{4})\s*$", mm_yy or "")
    if not m: return False
    mm = int(m.group(1))
    yy = int(m.group(2))
    if mm < 1 or mm > 12:
        return False
    if yy < 100:
        yy += 2000
    # consider expired end-of-month
    now = datetime.utcnow()
    return (yy > now.year) or (yy == now.year and mm >= now.month)

def mask_card(card_number: str):
    digits = re.sub(r"\D", "", card_number)
    last4 = digits[-4:] if len(digits) >= 4 else "xxxx"
    return f"**** **** **** {last4}", last4

def tokenize_card(card_number: str) -> str:
    # Stateless tokenization using HMAC digest (demo). In real-world use a PCI-compliant vault.
    cn = re.sub(r"\D", "", card_number)
    digest = hmac.new(APP_SECRET, cn.encode(), hashlib.sha256).digest()
    token = base64.urlsafe_b64encode(digest)[:28].decode()
    return f"TOK_{token}"

def sign_payment_payload(payload: dict) -> str:
    # Deterministic signature of JSON payload
    data = json.dumps(payload, separators=(",", ":"), sort_keys=True).encode()
    return hmac.new(GATEWAY_SECRET, data, hashlib.sha256).hexdigest()

# 3) Database setup
def db():
    return sqlite3.connect(DB_PATH, check_same_thread=False)

def init_db():
    con = db()
    cur = con.cursor()
    cur.executescript("""
    PRAGMA journal_mode=WAL;
    CREATE TABLE IF NOT EXISTS users(
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        email TEXT UNIQUE NOT NULL,
        salt TEXT NOT NULL,
        pw_hash TEXT NOT NULL,
        iters INTEGER NOT NULL,
        created_at TEXT NOT NULL
    );
    CREATE TABLE IF NOT EXISTS products(
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT NOT NULL,
        price_cents INTEGER NOT NULL,
        stock INTEGER NOT NULL DEFAULT 0
    );
    CREATE TABLE IF NOT EXISTS orders(
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        user_id INTEGER NOT NULL,
        total_cents INTEGER NOT NULL,
        status TEXT NOT NULL,
        created_at TEXT NOT NULL,
        FOREIGN KEY(user_id) REFERENCES users(id)
    );
    CREATE TABLE IF NOT EXISTS order_items(
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        order_id INTEGER NOT NULL,
        product_id INTEGER NOT NULL,
        qty INTEGER NOT NULL,
        price_cents INTEGER NOT NULL,
        FOREIGN KEY(order_id) REFERENCES orders(id),
        FOREIGN KEY(product_id) REFERENCES products(id)
    );
    CREATE TABLE IF NOT EXISTS payments(
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        order_id INTEGER NOT NULL,
        amount_cents INTEGER NOT NULL,
        status TEXT NOT NULL,
        tx_id TEXT NOT NULL,
        card_token TEXT,
        card_last4 TEXT,
        sig TEXT,            -- HMAC signature of payload
        created_at TEXT NOT NULL,
        FOREIGN KEY(order_id) REFERENCES orders(id)
    );
    """)
    con.commit()

    # Seed products once
    cur.execute("SELECT COUNT(*) FROM products")
    if cur.fetchone()[0] == 0:
        items = [
            ("Wireless Mouse", 8999, 20),
            ("Mechanical Keyboard", 55999, 10),
            ("USB-C Hub 7-in-1", 34999, 15),
            ("1080p Webcam", 42999, 12),
            ("Noise-Cancel Headset", 79999, 8),
        ]
        cur.executemany("INSERT INTO products(name, price_cents, stock) VALUES (?,?,?)", items)
        con.commit()
    con.close()

init_db()

# 4) Core operations
def register_user(email: str, password: str):
    email = (email or "").strip().lower()
    if not re.match(r"^[^@]+@[^@]+\.[^@]+$", email):
        return False, "Invalid email format."
    if not password or len(password) < 8:
        return False, "Password must be at least 8 characters."
    salt, pw_hash, iters = pbkdf2_hash(password)
    con = db()
    try:
        con.execute(
            "INSERT INTO users(email, salt, pw_hash, iters, created_at) VALUES(?,?,?,?,?)",
            (email, salt, pw_hash, iters, datetime.utcnow().isoformat())
        )
        con.commit()
        return True, "Registration successful. Please log in."
    except sqlite3.IntegrityError:
        return False, "Email already registered."
    finally:
        con.close()

def login_user(email: str, password: str):
    email = (email or "").strip().lower()
    con = db()
    cur = con.cursor()
    cur.execute("SELECT id, salt, pw_hash, iters FROM users WHERE email=?", (email,))
    row = cur.fetchone()
    con.close()
    if not row:
        return None, "User not found."
    uid, salt, pw_hash, iters = row
    if not pbkdf2_verify(password, salt, pw_hash, iters):
        return None, "Incorrect password."
    return uid, "Login successful."

def list_products():
    con = db()
    cur = con.cursor()
    cur.execute("SELECT id, name, price_cents, stock FROM products ORDER BY id")
    rows = cur.fetchall()
    con.close()
    return rows

def create_order(user_id: int, cart: dict):
    # cart: {product_id: qty}
    if not cart:
        return None, "Cart is empty."
    con = db()
    cur = con.cursor()
    # Validate stock and compute total
    total_cents = 0
    for pid, qty in cart.items():
        cur.execute("SELECT price_cents, stock FROM products WHERE id=?", (pid,))
        r = cur.fetchone()
        if not r:
            con.close()
            return None, f"Product {pid} not found."
        price, stock = r
        if qty < 1 or qty > stock:
            con.close()
            return None, f"Invalid qty for product {pid}. In stock: {stock}"
        total_cents += price * qty

    # Create order
    cur.execute(
        "INSERT INTO orders(user_id, total_cents, status, created_at) VALUES (?,?,?,?)",
        (user_id, total_cents, "PENDING", datetime.utcnow().isoformat()))
    order_id = cur.lastrowid

    # Create order_items and decrement stock
    for pid, qty in cart.items():
        cur.execute("SELECT price_cents FROM products WHERE id=?", (pid,))
        price = cur.fetchone()[0]
        cur.execute(
            "INSERT INTO order_items(order_id, product_id, qty, price_cents) VALUES (?,?,?,?)",
            (order_id, pid, qty, price))
        cur.execute("UPDATE products SET stock = stock - ? WHERE id=?", (qty, pid))

    con.commit()
    con.close()
    return order_id, total_cents

def mock_gateway_charge(order_id: int, amount_cents: int, card_token: str):
    # Simulate a payment gateway: sign a payload; randomly succeed (but mostly succeed).
    payload = {
        "order_id": order_id,
        "amount_cents": amount_cents,
        "card_token": card_token,
        "ts": int(time.time())
    }
    sig = sign_payment_payload(payload)
    # Make it deterministic success for demo
    tx_id = "TX_" + base64.urlsafe_b64encode(
        hmac.new(APP_SECRET, str(payload).encode(), hashlib.sha256).digest()
    )[:16].decode()
    return {"status": "success", "tx_id": tx_id, "sig": sig}

def capture_payment(order_id: int, amount_cents: int, card_number: str, expiry: str, cvv: str):
    # Validate inputs without storing PAN
    if not luhn_check(card_number):
        return False, "Card number failed Luhn check.", None
    if not validate_expiry(expiry):
        return False, "Invalid or expired date (use MM/YY).", None
    if not re.fullmatch(r"\d{3,4}", (cvv or "").strip()):
        return False, "Invalid CVV.", None

    token = tokenize_card(card_number)
    masked, last4 = mask_card(card_number)
    gw = mock_gateway_charge(order_id, amount_cents, token)

    con = db()
    cur = con.cursor()
    if gw["status"] == "success":
        cur.execute(
            "UPDATE orders SET status=? WHERE id=?",
            ("PAID", order_id)
        )
        cur.execute(
            "INSERT INTO payments(order_id, amount_cents, status, tx_id, card_token, card_last4, sig, created_at) "
            "VALUES (?,?,?,?,?,?,?,?)",
            (order_id, amount_cents, "CAPTURED", gw["tx_id"], token, last4, gw["sig"], datetime.utcnow().isoformat())
        )
        con.commit()
        con.close()
        receipt = {
            "order_id": order_id,
            "amount": f"₹{amount_cents/100:.2f}",
            "masked_card": masked,
            "tx_id": gw["tx_id"]
        }
        return True, f"Payment successful! Receipt:\n{json.dumps(receipt, indent=2)}", receipt
    else:
        cur.execute(
            "INSERT INTO payments(order_id, amount_cents, status, tx_id, created_at) VALUES (?,?,?,?,?)",
            (order_id, amount_cents, "DECLINED", "N/A", datetime.utcnow().isoformat())
        )
        con.commit()
        con.close()
        return False, "Payment declined by gateway.", None

def user_orders(user_id: int):
    con = db()
    cur = con.cursor()
    cur.execute("SELECT id, total_cents, status, created_at FROM orders WHERE user_id=? ORDER BY id DESC", (user_id,))
    orders = cur.fetchall()
    con.close()
    return orders

# 5) Gradio UI
def format_products():
    rows = list_products()
    lines = ["ID | Product | Price | Stock"]
    for pid, name, cents, stock in rows:
        lines.append(f"{pid:2d} | {name} | ₹{cents/100:.2f} | {stock}")
    return "\n".join(lines)

with gr.Blocks(title="Secure E-Commerce Demo (Python)") as demo:
    gr.Markdown(
        """
        # 🛡️ Secure E-Commerce (Python) — Demo
        **NOTE:** This is a **safe simulation**. No real payments.
        - Credentials stored with **PBKDF2-HMAC-SHA256 + salt**
        - Card **never stored**; a **token** is generated instead
        - Mock “gateway” uses **HMAC signatures**
        """
    )

    # Global State (per session)
    user_id_state = gr.State(value=None)
    email_state = gr.State(value=None)
    cart_state = gr.State(value={})  # {product_id: qty}

    with gr.Tab("Register / Login"):
        gr.Markdown("### Create account or login")
        reg_email = gr.Textbox(label="Email")
        reg_pw = gr.Textbox(label="Password (min 8 chars)", type="password")
        reg_btn = gr.Button("Register")
        reg_out = gr.Textbox(label="Registration status", interactive=False)

        log_email = gr.Textbox(label="Email")
        log_pw = gr.Textbox(label="Password", type="password")
        log_btn = gr.Button("Login")
        log_out = gr.Textbox(label="Login status", interactive=False)

        def do_register(e, p):
            ok, msg = register_user(e, p)
            return msg

        reg_btn.click(do_register, [reg_email, reg_pw], reg_out)

        def do_login(e, p):
            uid, msg = login_user(e, p)
            if uid:
                return msg, uid, e
            return msg, None, None

        log_btn.click(do_login, [log_email, log_pw], [log_out, user_id_state, email_state])

    with gr.Tab("Shop"):
        gr.Markdown("### Catalog")
        catalog = gr.Textbox(value=format_products(), lines=10, interactive=False)

        pid_in = gr.Number(label="Product ID", precision=0)
        qty_in = gr.Number(label="Quantity", precision=0, value=1)
        add_btn = gr.Button("Add to cart")
        add_out = gr.Textbox(label="Cart status", interactive=False)

        def add_to_cart(pid, qty, cart):
            try:
                pid = int(pid)
                qty = int(qty)
            except:
                return "Enter valid numeric ID/qty.", cart, format_products()
            if qty < 1:
                return "Quantity must be ≥ 1.", cart, format_products()
            # Check stock quickly
            rows = {r[0]: (r[1], r[2], r[3]) for r in list_products()}  # id -> (name, price, stock)
            if pid not in rows:
                return "Product ID not found.", cart, format_products()
            name, price, stock = rows[pid]
            if qty > stock:
                return f"Only {stock} in stock for {name}.", cart, format_products()
            cart = dict(cart or {})
            cart[pid] = cart.get(pid, 0) + qty
            return f"Added {qty} × {name} to cart.", cart, format_products()

        add_btn.click(add_to_cart, [pid_in, qty_in, cart_state], [add_out, cart_state, catalog])

    with gr.Tab("Cart & Checkout"):
        cart_view = gr.Textbox(label="Cart", lines=8, interactive=False)
        refresh_btn = gr.Button("Refresh Cart")

        def render_cart(cart):
            rows = {r[0]: (r[1], r[2]) for r in list_products()}  # id -> (name, price)
            total = 0
            lines = ["Your Cart:"]
            if not cart:
                return "Cart is empty."
            for pid, qty in cart.items():
                if pid in rows:
                    name, price = rows[pid]
                    subtotal = price * qty
                    total += subtotal
                    lines.append(f"{qty} × {name}  @ ₹{price/100:.2f} = ₹{subtotal/100:.2f}")
            lines.append(f"\nTotal: ₹{total/100:.2f}")
            return "\n".join(lines)

        refresh_btn.click(lambda c: render_cart(c), [cart_state], [cart_view])

        gr.Markdown("#### Payment (Demo — do not use real card info)")
        name_on_card = gr.Textbox(label="Name on card")
        card_num = gr.Textbox(label="Card number (test numbers only)")
        card_exp = gr.Textbox(label="Expiry (MM/YY)")
        card_cvv = gr.Textbox(label="CVV")

        checkout_btn = gr.Button("Pay Now (Mock Gateway)")
        pay_out = gr.Textbox(label="Payment result", lines=10, interactive=False)

        def do_checkout(user_id, cart, name, number, exp, cvv):
            if not user_id:
                return "Please log in first."
            # Create order
            oid, total_or_err = create_order(user_id, dict(cart or {}))
            if oid is None:
                return f"Order error: {total_or_err}"
            total_cents = total_or_err
            ok, msg, receipt = capture_payment(oid, total_cents, number or "", exp or "", cvv or "")
            if ok:
                # Clear cart on success
                return msg + "\n\n(Your cart has been cleared.)"
            else:
                return "Payment failed: " + msg

        checkout_btn.click(
            do_checkout,
            [user_id_state, cart_state, name_on_card, card_num, card_exp, card_cvv],
            [pay_out]
        )

    with gr.Tab("My Orders"):
        orders_out = gr.Textbox(label="Orders", lines=12, interactive=False)
        reload_orders = gr.Button("Reload")

        def load_orders(uid):
            if not uid: return "Please log in."
            orders = user_orders(uid)
            if not orders: return "No orders yet."
            lines = []
            for (oid, total, status, created) in orders:
                lines.append(f"Order #{oid} | ₹{total/100:.2f} | {status} | {created}")
            return "\n".join(lines)

        reload_orders.click(load_orders, [user_id_state], [orders_out])

    with gr.Accordion("Security Notes (click to expand)", open=False):
        gr.Markdown("""
- Passwords: **PBKDF2-HMAC-SHA256** with per-user salt and 200k iterations
- Cards: **Never stored**. We derive a **token** (HMAC digest) and keep only **last4** for receipts
- Gateway: mock charge is **HMAC-signed**; app records signature with payment
- Validation: Luhn check, expiry format, CVV format
- Database: SQLite with tables for users, products, orders, order_items, payments
        """)

demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()


/usr/local/lib/python3.12/dist-packages/gradio/analytics.py:106: UserWarning: IMPORTANT: You are using gradio version 4.44.0, however version 4.44.1 is available, please upgrade. 
--------
  warnings.warn(


Running on public URL: https://cabbdcddc487ee6c5f.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


In [ ]:
!pip install gradio
